# Skewness

A lottery ticket and a car-insurance policy can be *mirror images* of each other: same expected value, same variance — yet living them feels completely different. One is "lose a little almost always, win big rarely"; the other is "win a little almost always, lose big rarely". The first number that can tell them apart is the **third moment**, and its standardized version is the **skewness**: $E\big[\big(\frac{X-\mu}{\sigma}\big)^3\big]$. It is based on the tenth video of Lesson 3 (Describing Distributions).

## 1. Introduction

Two scenarios, with deliberately matched numbers:

- **You play a lottery.** A ticket costs \$1 and the jackpot is \$100. With 1% probability you win (net **+\$99**); with 99% probability you lose your dollar (**−\$1**).
- **You own a car-insurance company** with one client. The policy costs \$1. With 99% probability the client doesn't crash and you keep the dollar (**+\$1**); with 1% probability they crash and you pay \$100 for repairs (net **−\$99**).

The two games are exact opposites — one distribution is the other reflected over the vertical axis. Yet, as we will compute, the mean *and* the variance are identical. Telling them apart needs a new descriptor, and it matters: the difference between "rare huge gain" and "rare huge loss" is the difference between a lottery player's and an insurer's risk profile.

After completing this notebook you will be able to:

- Show that mirror-image distributions share their first and second moments.
- Use the **third moment** $E[X^3]$ to detect asymmetry, and explain *why* the cube works.
- Define **skewness** as the third moment of the standardized variable and classify distributions as positively skewed, symmetric, or negatively skewed.

## 2. Theory

### Two games the mean cannot tell apart

Lottery $X_1$ and insurance $X_2$:

| | outcome | probability |
|---|---|---|
| $X_1$ (lottery) | $-1$ | $0.99$ |
| | $+99$ | $0.01$ |
| $X_2$ (insurance) | $+1$ | $0.99$ |
| | $-99$ | $0.01$ |

$$
E[X_1] = (-1)(0.99) + (99)(0.01) = 0
\qquad
E[X_2] = (1)(0.99) + (-99)(0.01) = 0
$$

Both games break even on average: play the lottery many times, or sell many policies, and the long-run average is \$0 either way. The expected value fails to distinguish them.

### The variance fails too

The distributions are reflections of each other, so their spreads must match. Since both are centered ($\mu = 0$), the variance is just $E[X^2]$ — and squaring erases signs:

$$
\mathrm{Var}(X_1) = (-1)^2(0.99) + (99)^2(0.01) = 0.99 + 98.01 = 99 = \mathrm{Var}(X_2)
$$

Same first moment, same second moment. Vastly different games.

### The third moment to the rescue

Cubing *keeps* signs. Compute $E[X^3]$ for both:

$$
E[X_1^3] = (-1)^3(0.99) + (99)^3(0.01) = -0.99 + 9702.99 = 9702
$$

$$
E[X_2^3] = (1)^3(0.99) + (-99)^3(0.01) = 0.99 - 9702.99 = -9702
$$

Finally a difference — and an interpretable one. The calculation is dominated by the far-away value ($99^3 = 970{,}299$ is enormous): a value far to the **right** drives $E[X^3]$ strongly **positive**; a value far to the **left** drives it strongly **negative**. The cube detects whether the distribution has values *skewed* toward the right or the left.

Continuous version: a distribution with a long tail to the right ("**positively skewed**" — more far-out values to the right) has a large positive $E[X^3]$; a long left tail ("**negatively skewed**") gives a large negative one.

### Skewness: the standardized third moment

$E[X^3]$ almost is the measure we want — but it still depends on units and scale (our 9702 would change if the game were in cents). The fix is the one from the standardization notebook: center and scale first. The **skewness** is the third moment of the standardized variable:

$$
\text{skewness} = E\left[\left(\frac{X - \mu}{\sigma}\right)^3\right]
$$

Three cases:

- **Positively skewed** (long tail to the right): skewness $> 0$.
- **Symmetric**: skewness $= 0$.
- **Negatively skewed** (long tail to the left): skewness $< 0$.

Moral of the story: **when the first and second moments don't help, go to the third** — it tells you the skewness of the distribution.

## 3. Worked Examples

### Example 1: skewness of the lottery and the insurance

Both games have $\mu = 0$ and $\sigma = \sqrt{99}$, so

$$
\text{skew}(X_1) = \frac{E[X_1^3]}{\sigma^3} = \frac{9702}{99^{3/2}} \approx 9.85
\qquad
\text{skew}(X_2) = \frac{-9702}{99^{3/2}} \approx -9.85
$$

Large positive for the lottery (a heavy right tail: the rare jackpot), equally large negative for the insurer (the rare catastrophic payout). Mirror games get mirror skewness.

### Example 2: a symmetric game

The ±\$1 coin game ($\pm1$ with probability $\tfrac12$ each): $E[X^3] = \tfrac12(-1) + \tfrac12(1) = 0$, so skewness $= 0$. Symmetry makes every odd moment vanish — positive and negative deviations cancel in pairs.

### Example 3: the video's warm-up variable

$X$: $-2$ w.p. $\tfrac13$, $0$ w.p. $\tfrac16$, $1$ w.p. $\tfrac12$ (from the moments notebook). We computed $\mu = -\tfrac16$, $\mathrm{Var} = \tfrac{65}{36}$, and third *central* moment $E[(X-\mu)^3] = -1.2593$. So

$$
\text{skew}(X) = \frac{-1.2593}{(65/36)^{3/2}} \approx -0.52
$$

— mildly negatively skewed: the $-2$ sits farther from the mean than anything on the right.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

### The lottery vs. the insurance: moments one, two, three

In [ ]:
def moment(values, pmf, k):
    return np.sum(pmf * values.astype(float)**k)

def skewness(values, pmf):
    mu = moment(values, pmf, 1)
    sigma = np.sqrt(np.sum(pmf * (values - mu)**2))
    return np.sum(pmf * ((values - mu) / sigma)**3)

lottery_values = np.array([-1, 99])
lottery_pmf = np.array([0.99, 0.01])
insurance_values = np.array([1, -99])
insurance_pmf = np.array([0.99, 0.01])

print(f"{'':<12} {'E[X]':>6} {'Var':>6} {'E[X^3]':>9} {'skewness':>9}")
for name, v, p in [("lottery", lottery_values, lottery_pmf),
                   ("insurance", insurance_values, insurance_pmf)]:
    mean = moment(v, p, 1)
    var = moment(v, p, 2) - mean**2
    m3 = moment(v, p, 3)
    print(f"{name:<12} {mean:>6.2f} {var:>6.2f} {m3:>9.0f} {skewness(v, p):>9.3f}")

print("\nFirst and second moments identical; the third moment (and skewness)")
print("finally separates the two games.")

### Living the two games: simulated bankrolls

Identical mean and variance, opposite experience: the lottery player bleeds slowly and occasionally jumps up; the insurer earns steadily and occasionally falls off a cliff.

In [ ]:
rng = np.random.default_rng(42)

n_rounds = 2000
lottery_outcomes = rng.choice([-1, 99], p=[0.99, 0.01], size=n_rounds)
insurance_outcomes = rng.choice([1, -99], p=[0.99, 0.01], size=n_rounds)

plt.figure(figsize=(10, 5))
plt.plot(np.cumsum(lottery_outcomes), color="tab:blue",
         label="lottery player (skew +9.85)")
plt.plot(np.cumsum(insurance_outcomes), color="tab:orange",
         label="insurance company (skew -9.85)")
plt.axhline(0, color="black", linewidth=1)
plt.title("Cumulative winnings: same mean and variance, mirror-image experience")
plt.xlabel("round")
plt.ylabel("cumulative winnings ($)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### A skewness function on samples, checked against known shapes

In [ ]:
rng = np.random.default_rng(42)

def sample_skewness(x):
    z = (x - x.mean()) / x.std()
    return np.mean(z**3)

samples = {
    "normal (symmetric)": rng.normal(10, 2, size=500_000),
    "uniform (symmetric)": rng.uniform(0, 60, size=500_000),
    "exponential (right tail)": rng.exponential(5, size=500_000),
    "MINUS exponential (left tail)": -rng.exponential(5, size=500_000),
}

for name, x in samples.items():
    print(f"{name:<30} skewness = {sample_skewness(x):>7.3f}")
print("\n(The exponential's exact skewness is 2; symmetric shapes give 0.)")

### Skewness is scale-free

Because of the standardization inside, changing units cannot change skewness — only a reflection flips its sign.

In [ ]:
rng = np.random.default_rng(42)
x = rng.exponential(5, size=500_000)

print(f"skew(X)          = {sample_skewness(x):.3f}")
print(f"skew(100X + 7)   = {sample_skewness(100 * x + 7):.3f}   (same: units don't matter)")
print(f"skew(-X)         = {sample_skewness(-x):.3f}   (reflection flips the sign)")

## 5. Visualizations

### The two games are reflections of each other

The video's twin plots: each distribution has a tall bar near zero and a tiny far-away bar — on opposite sides.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

axes[0].bar([-1, 99], [0.99, 0.01], width=6, color="tab:blue", edgecolor="black")
axes[0].set_title("lottery: skew = +9.85\n(rare big win, far RIGHT)")
axes[1].bar([1, -99], [0.99, 0.01], width=6, color="tab:orange", edgecolor="black")
axes[1].set_title("car insurance: skew = -9.85\n(rare big loss, far LEFT)")

for ax in axes:
    ax.axvline(0, color="tab:red", linestyle="--", linewidth=1.5, label="mean = 0")
    ax.set_xlabel("net winnings ($)")
    ax.set_xlim(-115, 115)
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("probability")
plt.suptitle("Mirror games: same mean (0), same variance (99), opposite third moment (±9702)")
plt.tight_layout()
plt.show()

### Positive vs. negative skew, continuous version

The video's closing picture: a long right tail gives positive $E[X^3]$ (and skewness), a long left tail gives negative.

In [ ]:
rng = np.random.default_rng(42)

right_skewed = rng.exponential(2, size=500_000)          # long tail to the right
left_skewed = 10 - rng.exponential(2, size=500_000)      # mirrored: long left tail

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, data, color, label in [
        (axes[0], right_skewed, "tab:blue", "positively skewed\nskewness > 0"),
        (axes[1], left_skewed, "tab:orange", "negatively skewed\nskewness < 0")]:
    ax.hist(data, bins=100, density=True, color=color, alpha=0.7)
    ax.axvline(data.mean(), color="tab:red", linestyle="--", linewidth=2,
               label=f"mean (skew = {sample_skewness(data):+.2f})")
    ax.set_title(label)
    ax.set_xlabel("value")
    ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("density")
plt.suptitle("The sign of the third moment tells the direction of the tail")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** A game pays $+3$ with probability $\tfrac14$ and $-1$ with probability $\tfrac34$. Compute the mean, variance, third central moment, and skewness. Which way does it lean?

**Exercise 2.** Without computing anything: what is the skewness of a fair die roll, and why?

### Intermediate

**Exercise 3.** For a Bernoulli variable ($1$ w.p. $p$, $0$ w.p. $1-p$) the skewness is $\frac{1 - 2p}{\sqrt{p(1-p)}}$. Verify numerically for $p = 0.1, 0.3, 0.5, 0.7, 0.9$ and interpret the sign as a function of $p$.

**Exercise 4.** Reduce the lottery's jackpot from \$100 to \$2 (ticket still \$1, win probability still 1%). Recompute mean, variance, and skewness. The mean and variance collapse — but what happens to the skewness, and why? (Hint: any two-outcome game with these probabilities is a shifted and scaled Bernoulli.)

### Advanced

**Exercise 5.** Median vs. mean, revisited: for a positively skewed distribution the mean typically sits *right* of the median (the tail pulls it). Verify on the exponential distribution, and check the mirror statement on the negatively skewed version. Connect this to the median-and-mode notebook's Michael Jordan example — which way was that salary distribution skewed?

**Exercise 6.** Skewness of a sum: simulate the average of $n$ i.i.d. exponential variables (skewness 2 each) for $n = 1, 4, 16, 64$ and compute the skewness of the average. It should shrink like $2/\sqrt{n}$. What does this say about why averages of many effects tend to look symmetric (and eventually normal)?

## 7. Solutions

In [ ]:
# Exercise 1: +3 w.p. 1/4, -1 w.p. 3/4
values = np.array([3, -1])
pmf = np.array([0.25, 0.75])
mu = moment(values, pmf, 1)
var = moment(values, pmf, 2) - mu**2
m3c = np.sum(pmf * (values - mu)**3)
print(f"Exercise 1: mean = {mu}, variance = {var}, 3rd central moment = {m3c:.3f}")
print(f"skewness = {skewness(values, pmf):.3f} > 0: it leans right -- the rare +3")
print("sits farther from the mean than the common -1.")

In [ ]:
# Exercise 2: the fair die
faces = np.arange(1, 7)
pmf6 = np.full(6, 1 / 6)
print(f"Exercise 2: skewness = {skewness(faces, pmf6):.4f}")
print("Zero, with no computation needed: the die's PMF is symmetric around 3.5,")
print("so every deviation +d pairs with -d and all odd central moments cancel.")

In [ ]:
# Exercise 3: Bernoulli skewness (1 - 2p) / sqrt(p(1-p))
print("Exercise 3:")
print(f"{'p':>5} {'computed':>9} {'formula':>9}")
for p in [0.1, 0.3, 0.5, 0.7, 0.9]:
    values = np.array([0, 1])
    pmf = np.array([1 - p, p])
    formula = (1 - 2 * p) / np.sqrt(p * (1 - p))
    print(f"{p:>5.1f} {skewness(values, pmf):>9.3f} {formula:>9.3f}")
print("Small p: the rare 1 lies far right -> positive skew (like the lottery).")
print("Large p: the rare 0 lies far left -> negative skew (like the insurer).")
print("p = 0.5: symmetric, skewness 0.")

In [ ]:
# Exercise 4: shrinking the jackpot to $2
values = np.array([-1, 1])            # jackpot $2 - $1 ticket = net +1, w.p. 1%
pmf = np.array([0.99, 0.01])
mu = moment(values, pmf, 1)
var = moment(values, pmf, 2) - mu**2
print(f"Exercise 4: mean = {mu:.2f}, variance = {var:.4f}, "
      f"skewness = {skewness(values, pmf):.3f}")
print("Mean and variance collapsed (this game now LOSES $0.98 per ticket) -- but")
print("the skewness is EXACTLY the same 9.85 as before! Every two-outcome game")
print("with probabilities 0.99/0.01 is just aX + b applied to a Bernoulli(0.01),")
print("and skewness is standardized: shifts and scales cannot touch it. Its 9.85")
print("comes purely from the LOPSIDED PROBABILITIES, (1-2p)/sqrt(p(1-p)) at p=0.01.")

In [ ]:
# Exercise 5: skew direction vs mean-median order
rng = np.random.default_rng(0)

right = rng.exponential(5, size=1_000_000)
left = 20 - right

print("Exercise 5:")
print(f"  right-skewed: mean = {right.mean():.3f} > median = {np.median(right):.3f}")
print(f"  left-skewed:  mean = {left.mean():.3f} < median = {np.median(left):.3f}")
print("The tail drags the mean past the median in the tail's direction. The UNC")
print("salaries were positively (right) skewed: one enormous value far right,")
print("mean $250k >> median $22k.")

In [ ]:
# Exercise 6: skewness of averages shrinks like 2/sqrt(n)
rng = np.random.default_rng(0)

print("Exercise 6:")
print(f"{'n':>4} {'skew of average':>16} {'2/sqrt(n)':>10}")
for n in [1, 4, 16, 64]:
    averages = rng.exponential(1, size=(400_000, n)).mean(axis=1)
    print(f"{n:>4} {sample_skewness(averages):>16.3f} {2/np.sqrt(n):>10.3f}")
print("Averaging washes asymmetry out at rate 1/sqrt(n): sums of many small")
print("independent effects become symmetric -- the road to the normal distribution.")

## 8. Conclusion

- The lottery ($+99$ w.p. 1%, $-1$ w.p. 99%) and the insurance business ($-99$ w.p. 1%, $+1$ w.p. 99%) are mirror games: both have mean 0 **and** variance 99 — the first two moments cannot tell them apart.
- The **third moment** can, because cubing preserves signs while amplifying far-away values: $E[X_1^3] = 9702$ vs. $E[X_2^3] = -9702$ — dominated entirely by the $\pm 99^3$ term.
- **Skewness** is the standardized third moment, $E\big[\big(\frac{X-\mu}{\sigma}\big)^3\big]$: positive for right-skewed (long right tail), zero for symmetric, negative for left-skewed distributions. Standardizing makes it unit-free; reflections flip its sign.
- Skewness is the mathematical name for a real financial distinction: "rare big win" (lottery, positive skew) vs. "rare big loss" (insurance, negative skew).
- Moral: when the first and second moments don't help, go to the third.

Next up: the **fourth** moment and what it measures — kurtosis, the weight of the tails.